# 29. 다변량 분석: 가격·언어·태그의 복합 효과 검증

## 분석 목적

Slide 24(가격), Slide 25(태그), Slide 25-3(언어 지원)에서 각 요인을 **개별적으로** 분석했다.  
이 노트북은 세 요인을 **동시에 투입한 로지스틱 회귀**를 통해 아래 질문에 답한다.

1. 개별 요인이 복합적으로 투입될 때 각각의 독립적 기여도(오즈비)는 얼마인가?
2. 복합 모델이 단일 요인 모델보다 예측력이 높은가?
3. 슬라이드에서 제시한 언어 지원의 '약한 효과(Cramér\'s V=0.16)'가 다변량 통제 후에도 유지되는가?

## 분석 구조

| 단계 | 내용 |
|---|---|
| 1 | 데이터 로드 및 특성 생성 |
| 2 | 개별 요인 단변량 로지스틱 회귀 (기준선) |
| 3 | 복합 로지스틱 회귀 (가격 + 언어 + 태그 + 장르 통제) |
| 4 | 모델 성능 비교 (Pseudo R², AUC) |
| 5 | 오즈비 시각화 및 해석 |

In [ ]:
import re
import warnings
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

import statsmodels.api as sm
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

BASE_PATH = '../../../../data/preprocessed/'
RAW_PATH  = '../../../../data/raw/'

## 1. 데이터 로드 및 특성 생성

In [ ]:
df_graded  = pd.read_csv(os.path.join(BASE_PATH, 'steam_indie_games_graded.csv'))
df_silence = pd.read_csv(os.path.join(BASE_PATH, 'steam_indie_games_silence.csv'))
df_details = pd.read_csv(os.path.join(RAW_PATH, 'public_steam_app_details.csv'), usecols=['appid', 'supported_languages'])

df = pd.concat([df_graded, df_silence], ignore_index=True)
df.drop_duplicates(subset=['appid'], inplace=True)

# Stage 1(1~9개), Stage 2(10~49개)만 분석 대상
df = df[df['total_reviews'] > 0].copy()
df['stage'] = df['total_reviews'].apply(lambda x: 1 if x <= 9 else (2 if x <= 49 else 3))
df = df[df['stage'].isin([1, 2])].copy()
df['is_stage2'] = (df['stage'] == 2).astype(int)

print(f'분석 대상: {len(df):,}개 게임')
print(f'  Stage 1 (1~9 리뷰):  {(df["stage"]==1).sum():,}개 ({(df["stage"]==1).mean()*100:.1f}%)')
print(f'  Stage 2 (10~49 리뷰): {(df["stage"]==2).sum():,}개 ({(df["stage"]==2).mean()*100:.1f}%)')

### 1-1. 가격 특성

In [ ]:
df['price'] = pd.to_numeric(df['price'], errors='coerce').fillna(0)

# 기준 범주: $0-10 (가장 많은 게임이 몰린 구간)
df['price_10_20'] = ((df['price'] > 10) & (df['price'] <= 20)).astype(int)
df['price_20_30'] = ((df['price'] > 20) & (df['price'] <= 30)).astype(int)
df['price_30_plus'] = (df['price'] > 30).astype(int)

print('가격대 분포:')
bins   = [-1, 10, 20, 30, float('inf')]
labels = ['$0-10', '$10-20', '$20-30', '$30+']
df['price_range'] = pd.cut(df['price'], bins=bins, labels=labels)
print(df['price_range'].value_counts().sort_index())

### 1-2. 언어 지원 특성

In [ ]:
def count_languages(lang_str: str) -> int:
    """Steam supported_languages HTML 문자열에서 지원 언어 수를 반환한다."""
    if pd.isna(lang_str):
        return 0
    s = re.sub(r'<br\s*/?>', ', ', lang_str)
    s = re.sub(r'<[^>]+>', '', s)
    parts = [p.strip().strip('* ') for p in s.split(',')]
    langs = [p for p in parts if p and len(p) > 1 and 'languages with' not in p.lower()]
    return len(langs)

df = df.merge(df_details, on='appid', how='left')
df['lang_count'] = df['supported_languages'].apply(count_languages)

# 기준 범주: 영어 단독(1개)
df['lang_2_4'] = ((df['lang_count'] >= 2) & (df['lang_count'] <= 4)).astype(int)
df['lang_5plus'] = (df['lang_count'] >= 5).astype(int)

print('언어 지원 수 분포 (lang_count 결측 → 0으로 처리):')
bins_l   = [0, 1, 4, float('inf')]
labels_l = ['1개 (영어 단독)', '2~4개', '5개 이상']
df['lang_group'] = pd.cut(df['lang_count'], bins=bins_l, labels=labels_l, right=True)
print(df['lang_group'].value_counts().sort_index())
print(f'\nlang_count 결측(언어 데이터 없음): {df["lang_count"].isna().sum()}건')

### 1-3. 태그 특성

Part 2 분석(Slide 25-1)에서 무반응 그룹에 집중된 태그를 바이너리 특성으로 변환한다.  
OR < 0.5이고 무반응 보유율 상위였던 태그를 중심으로 선택한다.

In [ ]:
def parse_tag_keys(value) -> list:
    if pd.isna(value):
        return []
    try:
        parsed = json.loads(str(value))
        if isinstance(parsed, dict):
            return [str(k).strip() for k in parsed.keys()]
    except Exception:
        pass
    return []

df['tag_list'] = df['tags'].apply(parse_tag_keys)

# 무반응 집중 태그 (Slide 25-1 기준 OR < 0.5)
TARGET_TAGS = ['Singleplayer', 'Casual', '2D', '3D', 'Indie']

for tag in TARGET_TAGS:
    col = 'tag_' + tag.lower().replace(' ', '_')
    df[col] = df['tag_list'].apply(lambda t: int(tag in t))

tag_cols = ['tag_' + t.lower().replace(' ', '_') for t in TARGET_TAGS]
print('태그 보유율 (전체 기준):')
print(df[tag_cols].mean().round(3).rename(lambda x: x.replace('tag_', '')))

### 1-4. 장르 특성 (통제 변수)

In [ ]:
df['genres_clean'] = df['genres'].astype(str).str.replace(r"[\[\]\'\"\.]", '', regex=True)
MAJOR_GENRES = ['Action', 'Adventure', 'Casual', 'RPG', 'Simulation', 'Strategy']

# 기준 범주: Action
for g in MAJOR_GENRES[1:]:
    df['genre_' + g.lower()] = df['genres_clean'].str.contains(g, case=False, na=False).astype(int)

genre_cols = ['genre_' + g.lower() for g in MAJOR_GENRES[1:]]
print('장르 보유율 (기준 범주: Action):')
print(df[genre_cols].mean().round(3).rename(lambda x: x.replace('genre_', '')))

## 2. 단변량 로지스틱 회귀 (기준선)

각 요인 그룹을 개별적으로 투입해 기준 성능을 측정한다.

In [ ]:
def fit_logit(feature_cols, df, label):
    """statsmodels 로지스틱 회귀 적합 및 AUC 반환"""
    subset = df[feature_cols + ['is_stage2']].dropna()
    X = sm.add_constant(subset[feature_cols])
    y = subset['is_stage2']
    model = sm.Logit(y, X).fit(disp=False)
    pred  = model.predict(X)
    auc   = roc_auc_score(y, pred)
    pr2   = model.prsquared  # McFadden Pseudo R²
    print(f'[{label}]  AUC={auc:.4f}  Pseudo R²={pr2:.4f}  N={len(subset):,}')
    return model, auc, pr2

price_cols = ['price_10_20', 'price_20_30', 'price_30_plus']
lang_cols  = ['lang_2_4', 'lang_5plus']

print('=== 단변량 모델 AUC / Pseudo R² ===')
m_price, auc_price, pr2_price = fit_logit(price_cols,  df, '가격만')
m_lang,  auc_lang,  pr2_lang  = fit_logit(lang_cols,   df, '언어만')
m_tag,   auc_tag,   pr2_tag   = fit_logit(tag_cols,    df, '태그만')
m_genre, auc_genre, pr2_genre = fit_logit(genre_cols,  df, '장르만')

## 3. 복합 로지스틱 회귀

가격 + 언어 + 태그 + 장르(통제)를 동시에 투입한다.

In [ ]:
all_features = price_cols + lang_cols + tag_cols + genre_cols

print('=== 복합 모델 ===')
m_full, auc_full, pr2_full = fit_logit(all_features, df, '복합(가격+언어+태그+장르)')

print()
print('=== 상세 결과 ===')
print(m_full.summary2())

## 4. 모델 성능 비교

In [ ]:
results = pd.DataFrame({
    '모델': ['가격만', '언어만', '태그만', '장르만', '복합 모델'],
    'AUC':        [auc_price, auc_lang, auc_tag, auc_genre, auc_full],
    'Pseudo R²':  [pr2_price, pr2_lang, pr2_tag, pr2_genre, pr2_full],
})

print(results.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#9DB8D2'] * 4 + ['#2D3E71']

axes[0].barh(results['모델'], results['AUC'], color=colors)
axes[0].axvline(0.5, color='gray', linestyle='--', linewidth=0.8, label='무작위 기준(0.5)')
axes[0].set_xlabel('AUC')
axes[0].set_title('모델별 AUC 비교')
axes[0].legend(fontsize=9)
for i, v in enumerate(results['AUC']):
    axes[0].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)

axes[1].barh(results['모델'], results['Pseudo R²'], color=colors)
axes[1].set_xlabel('McFadden Pseudo R²')
axes[1].set_title('모델별 Pseudo R² 비교')
for i, v in enumerate(results['Pseudo R²']):
    axes[1].text(v + 0.0002, i, f'{v:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 5. 오즈비 시각화

복합 모델에서 각 특성의 오즈비(OR)와 95% 신뢰구간을 시각화한다.  
OR > 1이면 Stage 2 진입 가능성 증가, OR < 1이면 감소.

In [ ]:
# 오즈비 및 95% CI 계산
or_df = pd.DataFrame({
    'feature': m_full.params.index,
    'OR':      np.exp(m_full.params),
    'OR_lo':   np.exp(m_full.conf_int()[0]),
    'OR_hi':   np.exp(m_full.conf_int()[1]),
    'pvalue':  m_full.pvalues,
}).query('feature != "const"').reset_index(drop=True)

# 가독성 좋은 이름으로 매핑
label_map = {
    'price_10_20':       '가격 $10~20 (vs $0~10)',
    'price_20_30':       '가격 $20~30 (vs $0~10)',
    'price_30_plus':     '가격 $30+ (vs $0~10)',
    'lang_2_4':          '언어 2~4개 (vs 영어 단독)',
    'lang_5plus':        '언어 5개+ (vs 영어 단독)',
    'tag_singleplayer':  '태그: Singleplayer',
    'tag_casual':        '태그: Casual',
    'tag_2d':            '태그: 2D',
    'tag_3d':            '태그: 3D',
    'tag_indie':         '태그: Indie',
    'genre_adventure':   '장르: Adventure (vs Action)',
    'genre_casual':      '장르: Casual (vs Action)',
    'genre_rpg':         '장르: RPG (vs Action)',
    'genre_simulation':  '장르: Simulation (vs Action)',
    'genre_strategy':    '장르: Strategy (vs Action)',
}
or_df['label'] = or_df['feature'].map(label_map).fillna(or_df['feature'])
or_df['sig']   = or_df['pvalue'].apply(lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else '')))

print(or_df[['label', 'OR', 'OR_lo', 'OR_hi', 'pvalue', 'sig']].round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

# 그룹별 색상
group_color = {}
for f in or_df['feature']:
    if f.startswith('price'):   group_color[f] = '#E07B54'
    elif f.startswith('lang'):  group_color[f] = '#4C72B0'
    elif f.startswith('tag'):   group_color[f] = '#55A868'
    else:                       group_color[f] = '#AAAAAA'

colors = [group_color[f] for f in or_df['feature']]
y_pos  = range(len(or_df))

ax.barh(y_pos, or_df['OR'], xerr=[
    or_df['OR'] - or_df['OR_lo'],
    or_df['OR_hi'] - or_df['OR'],
], color=colors, alpha=0.8, capsize=4, height=0.6)

ax.axvline(1.0, color='black', linewidth=1.0, linestyle='--')
ax.set_yticks(y_pos)
ax.set_yticklabels([f"{row.label}  {row.sig}" for _, row in or_df.iterrows()], fontsize=10)
ax.set_xlabel('오즈비 (OR)  ※ 1.0 기준선 = 차이 없음', fontsize=11)
ax.set_title('복합 모델 오즈비 (95% CI)', fontsize=13)

# 범례
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#E07B54', label='가격'),
    Patch(facecolor='#4C72B0', label='언어 지원'),
    Patch(facecolor='#55A868', label='태그'),
    Patch(facecolor='#AAAAAA', label='장르 (통제)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

## 6. 결론 및 해석

아래 셀은 결과를 자동으로 요약한다.

In [ ]:
print('=== 분석 결론 요약 ===')
print()

# 1. 모델 성능 비교
best_single = results.iloc[:-1].loc[results.iloc[:-1]['AUC'].idxmax()]
print(f'[모델 성능]')
print(f'  단변량 최고 AUC: {best_single["모델"]} = {best_single["AUC"]:.4f}')
print(f'  복합 모델 AUC:   {auc_full:.4f}  (차이: +{auc_full - best_single["AUC"]:.4f})')
print(f'  복합 모델 Pseudo R²: {pr2_full:.4f}')
print()

# 2. 각 요인별 독립 기여도
print('[요인별 유의성 (복합 모델 기준)]')
for _, row in or_df.iterrows():
    direction = '↑' if row['OR'] > 1 else '↓'
    sig_str = row['sig'] if row['sig'] else 'n.s.'
    print(f'  {row["label"]:35s}  OR={row["OR"]:.3f} {direction}  p={row["pvalue"]:.3f} {sig_str}')
print()

# 3. 언어 지원 검증
lang_rows = or_df[or_df['feature'].str.startswith('lang')]
lang_sig  = (lang_rows['pvalue'] < 0.05).any()
print(f'[언어 지원 효과]')
print(f'  복합 모델 통제 후 언어 지원 유의성: {"유의" if lang_sig else "비유의 — 언어 효과는 다른 요인으로 설명됨"}')
for _, r in lang_rows.iterrows():
    print(f'  {r["label"]}: OR={r["OR"]:.3f}, p={r["pvalue"]:.3f} {r["sig"]}')

## 해석 노트

### 모델 한계
- **스냅샷 기반**: Stage 1/2 분류는 현재 시점 단면 관측이며 시계열 전환을 추적한 것이 아니다.
- **교란 변수**: 마케팅 예산, 출시 타이밍, 개발자 SNS 등 미측정 변수가 존재한다.
- **다중 장르**: 한 게임이 여러 장르에 속해 장르 더미 간 독립성이 완전하지 않다.
- **인과 해석 불가**: OR은 연관성이며 인과관계를 나타내지 않는다.

### 슬라이드 활용 기준
- 복합 모델 AUC가 최고 단변량 AUC 대비 의미 있게 높다면 → **"여러 조건이 함께 작용한다"는 근거로 사용 가능**
- 언어 지원이 복합 모델에서 비유의로 나온다면 → **Slide 29-2의 톤 다운이 정당화됨**
- 가격 OR이 복합 모델에서도 유의하다면 → **Slide 32의 가격 효과는 다른 요인과 독립적임을 보완 근거로 사용 가능**